# Project Alpha: RL Bot v63p

## 📋 TODO & Roadmap
- **Data Policy:** Should we apply `handle_zeros_as_nan` globally to all matrices?
- **Feature Scaling:** Finalize the `get_agent_view` dual-view scaling logic.
- **Performance:** Move heavy logic (`analyze_4d_regime`) into `core` modular library.

<details>
<summary><b>📜 Version History (v59 - v63)</b></summary>

### v63
- Added **Metric Registry** (Strong-typed, dual-view).
- Fixed **Index Poisoning** bug (Dates in Ticker column).
- Unified **Perception** and **Execution** in Audit packs.
- Added 4 New Pillars: Return Autocorr, Range Position, OBV Divergence, Convexity.

### v62
- Fixed **Object Reference Mutation** bug in `compute_alpha_ensemble`.
- Added `PostBuildVerifier`, `DeepDiveDebugger`, and `ForensicExporter`.

### v61
- Verified all metrics with Excel parity.
- Refactored `AlphaEngine` into Orchestrator pattern.

### v60
- Converted notebook code to modular system.
- Added Volatility Regime plots.

### v59
- Implemented **Result Pattern** for error handling.
- Switched to logarithmic returns globally.
</details>

## I. Initialization & Environment

In [8]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import sys
import os
import gc
import pandas as pd
import numpy as np
import multiprocessing
import random
import re
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Optional
from types import SimpleNamespace
from IPython.display import display, clear_output
from tqdm.notebook import tqdm
import plotly.io as pio

def add_project_root_to_path():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            return path
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not find notebooks_RLVR directory")

add_project_root_to_path()

# 2. Display Settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.precision", 4)

# 3. Project Imports
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput, MarketObservation
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.logic import AlphaLogic
from core.paths import OUTPUT_DIR
from core.quant import QuantUtils, TickerEngine
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import STRATEGY_REGISTRY

# 4. Load Environment Paths
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"✓ Paths Initialized.\n  OHLCV: {DATA_PATH_OHLCV}\n  Indices: {DATA_PATH_INDICES}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Paths Initialized.
  OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
  Indices: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


## II. Data Pipeline

In [2]:
print("⏳ Loading DataFrames... takes ~6 minutes")
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

print("⚡ Generating Features...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    df_fed=df_fed,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

print("🚀 Unstacking Wide Matrices...")
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

⏳ Loading DataFrames... takes ~6 minutes
⚡ Generating Features...
⚡ Generating Decoupled Features (Benchmark: SPY)...
🚀 Unstacking Wide Matrices...
✅ Pipeline Complete. Tickers: 1580, Days: 16185


In [3]:
# === PERSISTENCE SANDBOX ===
features_df.to_parquet("output/features_df.parquet", index=True)
macro_df.to_parquet("output/macro_df.parquet", index=True)
df_close_wide.to_parquet("output/df_close_wide.parquet", index=True)
df_atrp_wide.to_parquet("output/df_atrp_wide.parquet", index=True)
df_trp_wide.to_parquet("output/df_trp_wide.parquet", index=True)

In [4]:
# # === RELOADING FROM PARQUET ===
# features_df = pd.read_parquet("output/features_df.parquet")
# macro_df = pd.read_parquet("output/macro_df.parquet")
# df_close_wide = pd.read_parquet("output/df_close_wide.parquet")
# df_atrp_wide = pd.read_parquet("output/df_atrp_wide.parquet")
# df_trp_wide = pd.read_parquet("output/df_trp_wide.parquet")

## III. The Analysis Suite

In [9]:
# SINGLE SOURCE OF TRUTH
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("🎯 Master AlphaEngine Ready.")

🎯 Master AlphaEngine Ready.


In [11]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2027-01-01"),
    lookback_period=189,
    holding_period=5,
    metric="OBV Divergence (5d)",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=100,
    debug=False,
)

analyzer1, _ = create_walk_forward_analyzer(engine, _inputs, universe_subset=None)
analyzer1.show()

⚠️ DATA/UI MISMATCH WARNING
Requested Decision Date: 2027-01-01 is not available.
The UI Decision Date input box is showing a date beyond available history.
REPLACING WITH LATEST AVAILABLE DATE: 2026-04-15


DEBUG: 942 stocks passed filters on 2026-04-15


In [ ]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df)
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

## IV. Deep Dives & Verification

In [ ]:
def verify_alpha_integrity(engine: AlphaEngine, ticker: str, date: pd.Timestamp):
    """Manual audit of the 4 New Pillars (Autocorr, Range, OBV, Convexity)."""
    alpha_matrix = engine.compute_alpha_matrix(decision_date=date, lookback_period=21)
    if ticker not in alpha_matrix.index:
        return print("❌ Ticker filtered out.")

    # --- Autocorr ---
    engine_val = alpha_matrix.loc[ticker, "Return Autocorr (15d)"]
    raw_rets = (
        engine.df_ohlcv_raw.xs(ticker, level="Ticker")["Adj Close"]
        .pct_change()
        .loc[:date]
        .tail(16)
    )
    manual_val = raw_rets.corr(raw_rets.shift(1))
    print(
        f"Autocorr (15d): Engine={engine_val:.6f} | Manual={manual_val:.6f} | Delta={abs(engine_val-manual_val):.2e}"
    )

    # --- Range ---
    engine_val = alpha_matrix.loc[ticker, "Range Position (20d)"]
    t_df = engine.df_ohlcv_raw.xs(ticker, level="Ticker").loc[:date].tail(20)
    manual_val = (t_df["Adj Close"].iloc[-1] - t_df["Adj Low"].min()) / (
        t_df["Adj High"].max() - t_df["Adj Low"].min()
    )
    print(
        f"Range Pos (20d): Engine={engine_val:.6f} | Manual={manual_val:.6f} | Delta={abs(engine_val-manual_val):.2e}"
    )

    print("🎯 Audit Complete.")


verify_alpha_integrity(engine, "NVDA", pd.Timestamp("2026-03-31"))

In [ ]:
@dataclass
class FourDRegime:
    regime: str
    signal: str
    confidence: float
    risk_flags: list
    raw_values: dict
    metadata: dict


def analyze_4d_regime(engine, ticker, date, mode="hybrid"):
    """Logic for classifying market state based on 4 pillars."""
    # ... Implementation details ... (Truncated for readability, keep original logic in file)
    pass  # Replace with full original function during reorganization execution


# result = analyze_4d_regime(engine, "NVDA", pd.Timestamp("2026-03-31"))

## V. Builders & Heavy Lifting

In [ ]:
# --- THE MARATHON BAKE ---
CHECKPOINT_DIR = OUTPUT_DIR / "alpha_cache_v1_checkpoints"
FINAL_FILE = OUTPUT_DIR / "AlphaCache_Master_2000-2026.parquet"

# ParallelFeatureBuilder.run_marathon(
#     master_engine=engine,
#     lookbacks=[10, 21, 63, 189],
#     start_date="2000-01-01",
#     checkpoint_dir=CHECKPOINT_DIR,
#     batch_size=50,
#     num_workers=max(1, multiprocessing.cpu_count() - 2),
# )

# final_cache_df = FeatureCubeStitcher.assemble(CHECKPOINT_DIR, FINAL_FILE)

## VI. Forensic Sandbox & Utilities

In [ ]:
def cleanup_ui():
    clear_output(wait=True)
    pio.renderers.default = None
    gc.collect()
    print("🧹 Memory Cleared.")


def active_engine_audit():
    gc.collect()
    engines = [obj for obj in gc.get_objects() if type(obj).__name__ == "AlphaEngine"]
    print(f"Active AlphaEngine Instances: {len(engines)}")


active_engine_audit()

In [ ]:
# print(features_df.info())
# display(features_df.head())
# display(df_indices.tail())

In [ ]:
################################
################################
################################
################################

In [12]:
result = SU.visualize_analyzer_structure(analyzer=analyzer1)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(17,))
[  2]   📈 benchmark_series (shape=(17,))
[  3]   🧮 normalized_plot_data (shape=(17, 3))
[  4]   📂 tickers (len=3)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]   📈 initial_weights (shape=(3,))
[  9]   📂 perf_metrics (len=24)
[ 10]     🔢 full_p_gain (float)
[ 11]     🔢 full_p_sharpe (float)
[ 12]     🔢 full_p_sharpe_atrp (float)
[ 13]     🔢 full_p_sharpe_trp (float)
[ 14]     🔢 lookback_p_gain (float)
[ 15]     🔢 lookback_p_sharpe (float)
[ 16]     🔢 lookback_p_sharpe_atrp (float)
[ 17]     🔢 lookback_p_sharpe_trp (float)
[ 18]     🔢 holding_p_gain (float)
[ 19]     🔢 holding_p_sharpe (float)
[ 20]     🔢 holding_p_sharpe_atrp (float)
[ 21]     🔢 holding_p_sharpe_trp (float)
[ 22]     🔢 full_b_gain (float)
[ 23]     🔢 full_b_sharpe (float)
[ 24]     🔢 full_b_sharpe_atrp (float)
[ 25]     🔢 full_b_sharpe_trp (float)
[ 26]     🔢 lookback_b_gain (flo

In [ ]:
df_snapshot = SU.peek(50, result)
df_snapshot.to_csv(OUTPUT_DIR / "df_snapshot.csv")

df_full_universe_ranking = SU.peek(51, result)
df_full_universe_ranking.to_csv(OUTPUT_DIR / "df_full_universe_ranking.csv")

In [14]:
SU.peek(1, result)

 📍 INDEX: [1]
 🏷️  NAME:  portfolio_series
 📂 PATH:  audit_pack -> portfolio_series



Date
2026-03-31    1.0000
2026-04-01    0.9988
2026-04-02    0.9958
2026-04-06    1.0025
2026-04-07    0.9920
2026-04-08    1.0164
2026-04-09    1.0182
2026-04-10    1.0198
2026-04-13    1.0280
2026-04-14    1.0370
2026-04-15    1.0311
2026-04-16    1.0252
2026-04-17    1.0517
2026-04-20    1.0497
2026-04-21    1.0380
2026-04-22    1.0278
2026-04-23    1.0268
dtype: float64

Date
2026-03-31    1.0000
2026-04-01    0.9988
2026-04-02    0.9958
2026-04-06    1.0025
2026-04-07    0.9920
2026-04-08    1.0164
2026-04-09    1.0182
2026-04-10    1.0198
2026-04-13    1.0280
2026-04-14    1.0370
2026-04-15    1.0311
2026-04-16    1.0252
2026-04-17    1.0517
2026-04-20    1.0497
2026-04-21    1.0380
2026-04-22    1.0278
2026-04-23    1.0268
dtype: float64

In [ ]:
# 1. Setup Filters
ticker_to_audit = "TSLA"

# 2. Export Calculated Features (The "Answers")
subset_df = features_df.loc[idx[ticker_to_audit, :], :]

# Derive date range from the retrieved TSLA features
start_date = subset_df.index.get_level_values(1).min()
end_date = subset_df.index.get_level_values(1).max()

subset_df.to_csv(
    OUTPUT_DIR / "features_audit_2025.csv", index=True, encoding="utf-8-sig"
)

# 3. Export Raw Price/Volume (The "Inputs")
# We include 'SPY' for Beta/IR calculations
all_tickers = [ticker_to_audit, "SPY"]
raw_ohlcv = df_ohlcv.loc[idx[all_tickers, start_date:end_date], :]
raw_ohlcv.to_csv(OUTPUT_DIR / "raw_price_audit.csv", index=True)

# 4. Export Macro Data (The "Market Context")
# Contains Mkt_Ret and Macro_Trend needed for IR_63 and Beta_63
raw_macro = macro_df.loc[start_date:end_date, :]
raw_macro.to_csv(OUTPUT_DIR / "raw_macro_audit.csv", index=True)

print(
    "✅ All 3 audit files generated. You can now replicate 'features_audit_2025.csv' using the 'raw' files in Excel."
)

In [ ]:
features_df.loc["TSLA"]

,ATR,ATRP,TRP,RSI,Mom_21,Consistency,IR_63,Beta_63,DD_21,AutoCorr_15,Ret_1d,Range_Pos_20,Slope_P_5,Slope_V_5,Convexity,RollingStalePct,RollMedDollarVol,RollingSameVolCount
Date,,,,,,,,,,,,,,,,,,
2010-06-29,NaN,0.0000,0.0000,50.0000,NaN,NaN,0.0000,1.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-06-30,0.4747,0.2988,0.2988,0.0000,NaN,NaN,0.0000,1.0000,0.0000,0.0000,-0.0025,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-01,0.4677,0.3194,0.2573,0.0000,NaN,NaN,0.0000,1.0000,0.0000,0.0000,-0.0785,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-02,0.4552,0.3556,0.2286,0.0000,NaN,NaN,0.0000,1.0000,0.0000,0.0000,-0.1257,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-06,0.4425,0.4120,0.2588,0.0000,NaN,0.0,0.0000,1.0000,0.0000,0.0000,-0.1609,NaN,-0.1004,-0.6060,0.0000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-17,15.7212,0.0392,0.0509,60.3103,0.0200,0.8,-0.0784,1.6655,0.0000,0.2679,0.0301,0.8798,0.0322,0.7243,0.0028,0.0,2.8928e+10,0.0
2026-04-20,15.9176,0.0406,0.0471,56.3573,0.0321,0.6,-0.0908,1.6720,-0.0203,0.1469,-0.0203,0.7671,0.0172,0.2860,-0.0152,0.0,2.8928e+10,0.0
2026-04-21,15.4042,0.0399,0.0226,53.5282,0.0502,0.4,-0.0818,1.6542,-0.0354,0.1332,-0.0155,0.6827,-0.0019,-0.2395,-0.0341,0.0,2.8928e+10,0.0
